In [2]:
import pandas as pd

# Initialize sample student data to resolve the NameError
data = {
    "Student_ID": ["S001", "S002", "S003"],
    "Engagement_Level": ["Low Engagement", "Medium Engagement", "High Engagement"],
    "Quiz_Score": [85, 92, 78],
    "Completion_Rate": [80, 95, 60]
}

segmented_df = pd.DataFrame(data)
display(segmented_df)

,Student_ID,Engagement_Level,Quiz_Score,Completion_Rate
0,S001,Low Engagement,85,80
1,S002,Medium Engagement,92,95
2,S003,High Engagement,78,60


In [3]:
SEGMENT_STYLE = {
    "Low Engagement": (
        "The student has LOW engagement so far. Use very simple language, short sentences, "
        "one concrete everyday example, and end with one encouraging sentence."
    ),
    "Medium Engagement": (
        "The student has MEDIUM engagement. Give a clear, moderately detailed answer, and end "
        "with one practical tip for applying the concept."
    ),
    "High Engagement": (
        "The student has HIGH engagement. Give a more detailed, technical answer, and end by "
        "pointing to a more advanced related concept worth exploring next."
    ),
}

GUARDRAIL_INSTRUCTION = (
    "Only recommend a module if the student's completed modules satisfy its prerequisites. "
    "Never claim something is in the lecture if it is not in the provided content."
)

PROMPT_TEMPLATE = """You are an AI course guide for students.

Use ONLY the lecture content below to answer the question.

Lecture Content:
{context}

Student Profile:
- Engagement Level: {segment}
- Quiz Score: {quiz_score}
- Completion Rate: {completion_rate}%

Student Question:
{question}

Personalization instruction: {style_instruction}
Guardrail: {guardrail}

If the answer is not present in the lecture content, say
"I could not find this information in the lecture."
"""

def get_student_profile(student_id: str) -> dict | None:
    """Fetches a student's profile from the segmented_df."""
    student_id = student_id.strip().upper()
    row = segmented_df[segmented_df["Student_ID"] == student_id]
    if row.empty:
        return None
    student_data = row.iloc[0]
    return {
        "Engagement_Level": student_data["Engagement_Level"],
        "Quiz_Score": student_data["Quiz_Score"],
        "Completion_Rate": student_data["Completion_Rate"],
    }

def build_prompt(student_profile: dict, context: str, question: str) -> str:
    segment = student_profile["Engagement_Level"]
    return PROMPT_TEMPLATE.format(
        context=context,
        segment=segment,
        quiz_score=student_profile["Quiz_Score"],
        completion_rate=student_profile["Completion_Rate"],
        question=question,
        style_instruction=SEGMENT_STYLE[segment],
        guardrail=GUARDRAIL_INSTRUCTION,
    )

# Preview a built prompt
_sample_profile = get_student_profile("S001")
print(build_prompt(_sample_profile, "Clustering groups similar data points.", "What is K-Means?"))


You are an AI course guide for students.

Use ONLY the lecture content below to answer the question.

Lecture Content:
Clustering groups similar data points.

Student Profile:
- Engagement Level: Low Engagement
- Quiz Score: 85
- Completion Rate: 80%

Student Question:
What is K-Means?

Personalization instruction: The student has LOW engagement so far. Use very simple language, short sentences, one concrete everyday example, and end with one encouraging sentence.
Guardrail: Only recommend a module if the student's completed modules satisfy its prerequisites. Never claim something is in the lecture if it is not in the provided content.

If the answer is not present in the lecture content, say
"I could not find this information in the lecture."

